<h1><center>Laboratorio 7: Ensamblaje, Optimización de Hiperparámetros e Interpretabilidad 🤖</center></h1>

<center><strong>MDS7202: Laboratorio de Programación Científica para Ciencia de Datos</strong></center>

---

### Cuerpo Docente

- Profesores: Pablo Badilla y Diego Cortez
- Auxiliares: Valentina Rojas y Melanie Peña
- Ayudantes: Javiera Arévalo, Tamara Carrasco y Ignacio Reyes

### Equipo: SUPER IMPORTANTE - notebooks sin nombre no serán revisados

- Nombre de alumno 1: Álvaro Sifuentes Tasayco
- Nombre de alumno 2: Sebastián Morales Castillo

---

### Reglas

- **Grupos de 2 personas**
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Prohibido copiar.
- Uso de LLM (Copilot, Claude, Antigravity, Cursor, etc.) restringido a consultas, documentación y corrección de errores.

## Temas a tratar

- Ensamblaje: Bagging (`RandomForest`), Boosting (`XGBoost`, `LightGBM`) y Stacking.
- Optimización de Hiperparámetros con `Optuna` y visualización interactiva con `optuna-dashboard`.
- Interpretabilidad global: `Permutation Feature Importance (PFI)`.
- Interpretabilidad local: `SHAP`.

### Objetivos principales del laboratorio

- Aplicar y comparar métodos de ensamblaje sobre un problema de clasificación de texto.
- Optimizar hiperparámetros de LightGBM usando Optuna y visualizar el proceso con `optuna-dashboard`.
- Interpretar las predicciones del modelo usando PFI y SHAP.

El laboratorio deberá ser desarrollado sin el uso indiscriminado de iteradores nativos de Python (aka "for", "while"). La idea es que aprendan a exprimir al máximo las funciones optimizadas que nos entrega `pandas`.

### Instalamos librerías 😸

In [16]:
!uv add nltk lightgbm xgboost optuna shap scikit-learn plotly

Resolved 146 packages in 31ms
Checked 138 packages in 125ms


In [ ]:
import warnings

import nltk
import optuna
import pandas as pd
import plotly.express as px
import shap
from lightgbm import LGBMClassifier
from nltk import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from optuna.visualization import (
    plot_optimization_history,
    plot_parallel_coordinate,
    plot_param_importances,
)
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

RANDOM_STATE = 42
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 1. ¿Quién es Bat Cow?

<p align="center">
  <img src="https://i.imgur.com/D9f1RHy.jpg" width="350">
</p>

En vez de estar desarrollando las evaluaciones correspondientes a su curso, su profesor de catedra y su auxiliar discuten acerca la alineación (i.e., si es heroe o villano) del personaje de ficción Bat-Cow.

El cuerpo docente, no logra ponerse de acuerdo si el personaje es bueno, neutral o malo: el auxiliar plantea que Bat-cow posee una siniestra mirada, intrigante pero común característica de los personajes malvados.
Por otra parte, extendiendo las ideas de Rousseau, el profesor plantea que tal como los humanos no nacen malos, no existe motivo por el cual una vaca con superpoderes deba serlo.

Sin embargo, ambos concuerdan que es difícil estimar la alineación solo usando los atributos físicos. Es por esto que les solicitan construir y optimizar un clasificador basado en texto que analice la alineación de cada personaje basado en su historia personal.

Para este laboratorio deben trabajar con los datos `df_comics.csv` y `comics_no_label.csv` subidos a u-cursos.

In [18]:
df_comics = pd.read_csv("df_comics.csv", index_col=0)
df_comics_no_label = pd.read_csv("comics_no_label.csv", index_col=0)
df_comics = df_comics.dropna(subset=["history_text"])
df_comics

,name,real_name,full_name,overall_score,history_text,powers_text,intelligence_score,strength_score,speed_score,durability_score,...,has_flight,has_accelerated_healing,has_weapons_master,has_intelligence,has_reflexes,has_super_speed,has_durability,has_stamina,has_agility,has_super_strength
0,3-D Man,"Delroy Garrett, Jr.","Delroy Garrett, Jr.",6,"Delroy Garrett, Jr. grew up to become a track ...",NaN,85,30,60,60,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
2,A-Bomb,Richard Milhouse Jones,Richard Milhouse Jones,20,"Richard ""Rick"" Jones was orphaned at a young ...","On rare occasions, and through unusual circu...",80,100,80,100,...,0.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0
3,Aa,Aa,NaN,12,Aa is one of the more passive members of the P...,NaN,80,50,55,45,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Aaron Cash,Aaron Cash,Aaron Cash,5,Aaron Cash is the head of security at Arkham A...,NaN,80,10,25,40,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,Aayla Secura,Aayla Secura,NaN,8,ayla Secura was a Rutian Twi'lek Jedi Knight (...,NaN,90,40,45,55,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1445,Zatanna,Zatanna Zatara,Zatanna Zatara,10,Zatanna is the daughter of adventurer John Zat...,Zatanna is genetically talented with her magi...,90,10,25,30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1446,Zero,DWN-∞: Zero,DWN-∞: Zero,18,Zero was created by the late Dr. Albert Wily ...,NaN,80,100,100,100,...,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1447,Zoom (New 52),Hunter Zolomon,NaN,20,"Hunter Zolomon is better known as Zoom, a spee...",After tricking Barry Allen and Wally West into...,95,50,100,75,...,0.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1448,Zoom,Hunter Zolomon,Hunter Zolomon,9,Hunter Zolomon had a troubled relationship wi...,"Zoom is able to alter time, to make himself ev...",75,10,100,30,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


## 1.1 Obtención de Features y Bag of Words

<p align="center">
  <img src="https://media0.giphy.com/media/eIUpSyzwGp0YhAMTKr/200.gif" width="300">
</p>

`bag of words` es un modelo de conteo utilizado en NLP que genera una representación vectorial para cada documento a través del conteo de las palabras que contienen.

<p align="center">
  <img src="https://user.oc-static.com/upload/2020/10/23/16034397439042_surfin%20bird%20bow.png" width="500">
</p>

Para facilitar el conteo transformamos cada documento en un vector mediante **tokenización**:

In [19]:
docs = ["The teacher rocks like a good rock & roll", "the rock is the best actor in the world"]
docs_tokenizados = [word_tokenize(doc) for doc in docs]
docs_tokenizados

[['The', 'teacher', 'rocks', 'like', 'a', 'good', 'rock', '&', 'roll'],
 ['the', 'rock', 'is', 'the', 'best', 'actor', 'in', 'the', 'world']]

Podemos mejorar la tokenización con:

- **Stemming**: transforma palabras a su forma raíz (*running → run*, *rocks → rock*).
- **Eliminación de Stopwords**: elimina palabras muy frecuentes que entorpecen la clasificación (*the*, *is*, *a*, ...).

<p align="center">
  <img src="https://devopedia.org/images/article/218/8583.1569386710.png" width="300">
</p>

In [20]:
stop_words = stopwords.words("english")


class StemmerTokenizer:
    def __init__(self):
        self.ps = PorterStemmer()

    def __call__(self, doc):
        doc_tok = word_tokenize(doc)
        doc_tok = [t for t in doc_tok if t not in stop_words]
        return [self.ps.stem(t) for t in doc_tok]


tokenizador = StemmerTokenizer()

docs = [
    "The teacher rocks like a good rock & roll",
    "the rock is the best actor in the world",
    "New York is a beautiful city",
]

print("Con StemmerTokenizer:")
print([tokenizador(doc) for doc in docs])
print("\nSin preprocesamiento:")
print([word_tokenize(doc) for doc in docs])

Con StemmerTokenizer:
[['the', 'teacher', 'rock', 'like', 'good', 'rock', '&', 'roll'], ['rock', 'best', 'actor', 'world'], ['new', 'york', 'beauti', 'citi']]

Sin preprocesamiento:
[['The', 'teacher', 'rocks', 'like', 'a', 'good', 'rock', '&', 'roll'], ['the', 'rock', 'is', 'the', 'best', 'actor', 'in', 'the', 'world'], ['New', 'York', 'is', 'a', 'beautiful', 'city']]


#### Al Estilo Scikit

Scikit implementa `bag of words` con `CountVectorizer()`. Además soporta **n-gramas**: secuencias contiguas de n palabras que se tratan como un único token. Esto permite capturar contexto local que los unigramas pierden.

| Tipo | n | Tokens de `"nueva york ciudad"` |
|------|---|--------------------------------|
| Unigrama | 1 | `nueva`, `york`, `ciudad` |
| Bigrama | 2 | `nueva york`, `york ciudad` |
| Trigrama | 3 | `nueva york ciudad` |

Con `ngram_range=(1,2)` el vectorizador incluye **unigramas y bigramas** simultáneamente. Los bigramas son especialmente útiles para capturar expresiones compuestas como `bat cow`, `spider man` o `super hero` que pierden su significado si se separan.

El parámetro `max_features` limita el vocabulario a los n tokens más frecuentes, controlando la dimensionalidad de la representación.

In [21]:
bow = CountVectorizer(tokenizer=StemmerTokenizer(), ngram_range=(1, 2))
df_bow = bow.fit_transform(docs)
pd.DataFrame(df_bow.toarray(), columns=bow.get_feature_names_out())

,&,& roll,actor,actor world,beauti,beauti citi,best,best actor,citi,good,...,rock,rock &,rock best,rock like,roll,teacher,teacher rock,world,york,york beauti
0,1,1,0,0,0,0,0,0,0,1,...,2,1,0,1,1,1,1,0,0,0
1,0,0,1,1,0,0,1,1,0,0,...,1,0,1,0,0,0,0,1,0,0
2,0,0,0,0,1,1,0,0,1,0,...,0,0,0,0,0,0,0,0,1,1


#### Combinando Features: `ColumnTransformer`

Para combinar en un solo paso el preprocesamiento de texto y numérico, usamos `ColumnTransformer`. Este aplica transformadores distintos a subconjuntos de columnas del DataFrame y concatena el resultado en una sola matriz de features lista para entrenar.

<p align="center">
  <img src="https://c.tenor.com/LkQzw7k5DV4AAAAd/anime-hacking.gif" width="300">
</p>

El `preprocessing_transformer` que usaremos a lo largo del lab combina:

- **`CountVectorizer`** con `StemmerTokenizer`, `ngram_range=(1,2)` y `max_features=500` → aplicado sobre la columna `history_text`.
- **`MinMaxScaler`** → aplicado sobre los 6 atributos numéricos de habilidad: `intelligence_score`, `strength_score`, `speed_score`, `durability_score`, `power_score`, `combat_score`.

In [ ]:
preprocessing_transformer = ColumnTransformer(
    transformers=[
        (
            "MinMaxScaler",
            MinMaxScaler(),
            [
                "intelligence_score",
                "strength_score",
                "speed_score",
                "durability_score",
                "power_score",
                "combat_score",
            ],
        ),
        (
            "bow",
            CountVectorizer(
                tokenizer=StemmerTokenizer(),
                max_features=500,
                ngram_range=(1, 2),
            ),
            "history_text",
        ),
    ]
)

## 1.2 Diseño de Baseline y Primer Entrenamiento [1 Punto]

<p align="center">
  <img src="https://pa1.narvii.com/6374/9eaec1b7bf9157334151452a669516f9a78b954c_hq.gif" width="300">
</p>

### 1.2.1 ¿Qué es un Baseline? [0.2 Puntos]

Antes de entrenar modelos complejos, es fundamental establecer un punto de referencia mínimo. Responde las siguientes preguntas con tus propias palabras:

1. **¿Qué es un baseline en Machine Learning?** ¿Para qué sirve establecerlo antes de evaluar modelos más sofisticados?
2. **¿Por qué usamos un `DummyClassifier` como baseline?** ¿Qué implica que un modelo "real" no logre superar su rendimiento?

> **Respuesta:**

**Respuesta:**

1. **Baseline.** Modelo trivial que fija el piso mínimo de desempeño. Sirve para confirmar que cualquier modelo más complejo realmente está aprendiendo algo y para detectar errores de pipeline o métrica.

2. **¿Por qué `DummyClassifier`?** No usa las features: solo replica la distribución de clases. Si un modelo "real" no lo supera, no está extrayendo señal útil del texto ni de los scores numéricos.


---

### 1.2.2 Implementación [0.6 Puntos]

Genere un `Pipeline` con las características de 1.1 y un `DecisionTreeClassifier()` por defecto.

Separe el dataset en entrenamiento/prueba (80/20, estratificado, `random_state=RANDOM_STATE`). Entrene, reporte `classification_report` y compare con un `DummyClassifier(strategy="stratified")`.

**To-do:**
- [ ] Pipeline con preprocesamiento → `DecisionTreeClassifier`.
- [ ] Holdout estratificado 80/20.
- [ ] `classification_report` del baseline.
- [ ] Entrenar `DummyClassifier` y comparar.

In [ ]:
FEATURES = [
    "history_text",
    "intelligence_score",
    "strength_score",
    "speed_score",
    "durability_score",
    "power_score",
    "combat_score",
]
TARGET = "alignment"

X = df_comics[FEATURES]
y = df_comics[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

pipe_tree = Pipeline(
    [
        ("features", preprocessing_transformer),
        ("clf", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ]
)

pipe_tree.fit(X_train, y_train)
y_pred_tree = pipe_tree.predict(X_test)

print("=== DecisionTreeClassifier ===")
print(classification_report(y_test, y_pred_tree))

pipe_dummy = Pipeline(
    [
        ("features", preprocessing_transformer),
        ("clf", DummyClassifier(strategy="stratified", random_state=RANDOM_STATE)),
    ]
)
pipe_dummy.fit(X_train, y_train)
y_pred_dummy = pipe_dummy.predict(X_test)

print("=== DummyClassifier (stratified) ===")
print(classification_report(y_test, y_pred_dummy))

KeyboardInterrupt: 

### 1.2.3 Pregunta de Cierre [0.2 Puntos]

**Pregunta:** ¿El `DecisionTreeClassifier` supera al `DummyClassifier`? ¿Qué concluyes de esto sobre lo que ha aprendido el modelo? Además responde:

1. ¿Por qué el accuracy puede ser una métrica engañosa en este problema? ¿Qué métrica es más apropiada si las clases están desbalanceadas?
2. ¿Por qué se usa el parámetro `stratify` en el `train_test_split`? ¿Qué problema evitamos al usarlo?
3. ¿Es mejor el clasificador que su versión aleatoria? ¿Podemos avanzar con confianza de que estamos clasificando mejor que si por ejemplo, tiraramos un dado con 3 caras?

> **Respuesta:**

**Respuesta:**

Sí, el `DecisionTreeClassifier` supera al `DummyClassifier` en accuracy y F1-Macro, así que aprendió algo de las features. La diferencia es modesta y motiva pasar a ensamblajes.

1. El dataset está desbalanceado (`Good` >> `Bad` > `Neutral`), así que un modelo que prediga siempre la mayoritaria puede tener accuracy alto sin aprender nada útil. **F1-Macro** es más apropiado: promedia el F1 de cada clase sin ponderar por frecuencia.

2. `stratify=y` preserva la proporción de clases en train y test. Sin esto, por azar el test podría quedar con muy pocas instancias de las clases minoritarias y la evaluación quedaría sesgada.

3. El árbol supera al dummy, pero la mejora es acotada. El `DummyClassifier(stratified)` ya es más exigente que un dado uniforme de 3 caras (respeta el desbalance), así que superarlo es necesario pero no suficiente.


---

# 2. Métodos de Ensamblaje [2 Puntos]

<p align="center">
  <img src="https://media.giphy.com/media/l0HlHFRbmaZtBRhXG/giphy.gif" width="300">
</p>

Los métodos de ensamblaje combinan múltiples modelos para obtener predicciones más robustas. Exploraremos tres estrategias:

| Estrategia | Idea clave | Ejemplo |
|------------|-----------|---------|
| **Bagging** | Modelos en paralelo sobre subconjuntos aleatorios | Random Forest |
| **Boosting** | Modelos en secuencia, cada uno corrige al anterior | XGBoost, LightGBM |
| **Stacking** | Predicciones de modelos base como input de un meta-modelo | StackingClassifier |

Todos usarán el mismo `preprocessing_transformer` de la sección 1.

## 2.1 Bagging: Random Forest [0.5 Puntos]

### 2.1.1 Descripción del algoritmo [0.2 Puntos]

Describe con tus propias palabras cómo funciona el **Bagging (Bootstrap Aggregating)**. La descripción debe cubrir los siguientes tres pasos:

1. **Generación de subconjuntos**: ¿Cómo se obtienen los subconjuntos de entrenamiento a partir del dataset original? ¿Se usa todo el dataset en cada uno? ¿Se pueden repetir instancias?
2. **Entrenamiento**: ¿Qué se entrena sobre cada subconjunto? ¿Los modelos se entrenan de forma dependiente o independiente entre sí?
3. **Agregación**: ¿Cómo se combinan las predicciones de todos los modelos para obtener una respuesta final?

> **Respuesta:**


**Respuesta:**

1. **Subconjuntos.** *Bootstrap*: muestreos aleatorios **con reemplazo** del tamaño original. Cada subconjunto puede repetir instancias y deja fuera ~36.8% (*out-of-bag*). `RandomForest` además muestrea features en cada split.

2. **Entrenamiento.** Un modelo base por subconjunto (un árbol en RF), entrenados de forma **independiente** entre sí. Permite paralelizar.

3. **Agregación.** Votación por mayoría (o promedio de probabilidades) en clasificación; promedio en regresión. Reduce varianza al promediar modelos con errores parcialmente descorrelacionados.


---

### 2.1.2 Implementación [0.2 Puntos]

**To-do:**
- [ ] Pipeline con `RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)`.
- [ ] Entrenar y reportar `classification_report`.

In [ ]:
pipe_rf = Pipeline(
    [
        ("features", preprocessing_transformer),
        (
            "clf",
            RandomForestClassifier(
                n_estimators=100,
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ]
)

pipe_rf.fit(X_train, y_train)
y_pred_rf = pipe_rf.predict(X_test)

print("=== RandomForestClassifier ===")
print(classification_report(y_test, y_pred_rf))

=== RandomForestClassifier ===
              precision    recall  f1-score   support

         Bad       0.60      0.30      0.40        86
        Good       0.63      0.91      0.75       148
     Neutral       0.00      0.00      0.00        23

    accuracy                           0.63       257
   macro avg       0.41      0.40      0.38       257
weighted avg       0.57      0.63      0.57       257



### 2.1.3 Pregunta de Cierre [0.1 Puntos]

**Pregunta:** ¿El Random Forest mejoró respecto al baseline? Comenta los resultados observados en el `classification_report` y explica a qué se debe la diferencia (o falta de ella), considerando las características del algoritmo que describiste anteriormente.

> **Respuesta:**

**Respuesta:**

Resultado mixto. El RF mejora accuracy respecto al árbol (`0.63` vs `0.51`) pero **empeora levemente F1-Macro** (`0.38` vs `0.39`):

- `Good`: recall sube fuerte (`0.91` vs `0.61`) → infla la accuracy.
- `Bad`: mejora precision pero pierde recall.
- `Neutral`: colapsa a `0.00`.

Es consistente con 2.1.1: el bagging suaviza las predicciones hacia la clase mayoritaria, y con clases minoritarias muy pequeñas, cada árbol tiene poca evidencia y la votación pierde. RF reduce varianza pero **no resuelve el desbalance**, lo que motiva pasar a Boosting.


## 2.2 Boosting: XGBoost y LightGBM [0.8 Puntos]

### 2.2.1 Descripción del algoritmo [0.3 Puntos]

Describe con tus propias palabras cómo funciona el **Boosting**. Tu descripción debe cubrir los siguientes tres pasos:

1. **Entrenamiento secuencial**: ¿En qué se diferencia el Boosting del Bagging en cuanto al orden en que se entrenan los modelos? ¿Son independientes entre sí?
2. **Corrección de errores**: ¿Cómo sabe cada modelo nuevo en qué instancias debe enfocarse? ¿Qué información del modelo anterior utiliza?
3. **Predicción final**: ¿Cómo se combinan las predicciones de todos los modelos? ¿Es una votación simple o una combinación ponderada?

Además, explica brevemente en qué se diferencian **XGBoost** y **LightGBM** como implementaciones de Boosting, y por qué XGBoost requiere que las etiquetas sean numéricas mientras que LightGBM acepta strings directamente.

> **Respuesta:**

**Respuesta:**

1. **Secuencial.** A diferencia del Bagging, los modelos se entrenan **uno tras otro**: el modelo `t+1` se condiciona al desempeño de los anteriores. No son independientes.

2. **Corrección.** Cada modelo nuevo se enfoca en los errores acumulados. En *gradient boosting* (XGBoost, LightGBM) cada árbol ajusta el **gradiente negativo de la pérdida** evaluado en las predicciones actuales: predice el residual que falta cerrar.

3. **Predicción final.** **Suma ponderada** por `learning_rate`, no votación. Es una combinación aditiva que construye iterativamente la función de decisión.

**XGBoost vs LightGBM.** Ambos son gradient boosting sobre árboles. XGBoost crece **level-wise** (preciso pero más lento); LightGBM crece **leaf-wise** (más rápido y eficiente, sobre todo en features sparsos como bag-of-words, pero más propenso a sobreajustar sin regularización).

**Etiquetas.** Internamente ambos necesitan target numérico. `LGBMClassifier` incluye un *label encoder* interno; `XGBClassifier` exige que las etiquetas lleguen ya codificadas como enteros, por eso usamos `LabelEncoder` + `inverse_transform`.


---

### 2.2.2 Implementación [0.4 Puntos]

**To-do:**
- [ ] Crear `LabelEncoder`, ajustarlo sobre `y_train` y transformar `y_train` e `y_test`.
- [ ] Pipeline con `XGBClassifier(random_state=RANDOM_STATE, eval_metric="mlogloss", verbosity=0)`. Reportar resultados decodificando las predicciones con `le.inverse_transform`.
- [ ] Pipeline con `LGBMClassifier(random_state=RANDOM_STATE, verbose=-1)`. Reportar resultados.

In [ ]:
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

pipe_xgb = Pipeline(
    [
        ("features", preprocessing_transformer),
        (
            "clf",
            XGBClassifier(
                random_state=RANDOM_STATE,
                eval_metric="mlogloss",
                verbosity=0,
                n_jobs=-1,
            ),
        ),
    ]
)

pipe_xgb.fit(X_train, y_train_enc)
y_pred_xgb_enc = pipe_xgb.predict(X_test)
y_pred_xgb = le.inverse_transform(y_pred_xgb_enc)

print("=== XGBClassifier ===")
print(classification_report(y_test, y_pred_xgb))

pipe_lgbm = Pipeline(
    [
        ("features", preprocessing_transformer),
        (
            "clf",
            LGBMClassifier(
                random_state=RANDOM_STATE,
                verbose=-1,
                n_jobs=-1,
            ),
        ),
    ]
)

pipe_lgbm.fit(X_train, y_train)
y_pred_lgbm = pipe_lgbm.predict(X_test)

print("=== LGBMClassifier ===")
print(classification_report(y_test, y_pred_lgbm))

=== XGBClassifier ===
              precision    recall  f1-score   support

         Bad       0.55      0.42      0.47        86
        Good       0.68      0.84      0.75       148
     Neutral       0.33      0.09      0.14        23

    accuracy                           0.63       257
   macro avg       0.52      0.45      0.45       257
weighted avg       0.60      0.63      0.60       257

=== LGBMClassifier ===
              precision    recall  f1-score   support

         Bad       0.60      0.56      0.58        86
        Good       0.72      0.86      0.78       148
     Neutral       0.00      0.00      0.00        23

    accuracy                           0.68       257
   macro avg       0.44      0.47      0.45       257
weighted avg       0.61      0.68      0.64       257



### 2.2.3 Pregunta de Cierre [0.1 Puntos]

**Pregunta:** Compara los resultados de XGBoost y LightGBM según el `classification_report`. ¿Cuál tuvo mejor desempeño en F1-Macro? ¿Ambos mejoran respecto al baseline? Considerando las diferencias que describiste en 2.2.1, ¿a qué atribuyes las similitudes o diferencias en rendimiento?

> **Respuesta:**

**Respuesta:**

Ambos superan al baseline y al `RandomForest` en F1-Macro (`0.45` vs `0.38`), confirmando que el Boosting empuja al modelo a corregir errores en clases minoritarias, no solo a votar.

- `Good`: F1 alto en ambos (XGB `0.75`, LGBM `0.78`).
- `Bad`: LGBM gana en recall (`0.56` vs `0.42`).
- `Neutral`: XGB al menos intenta predecirla (recall `0.09`); LGBM también colapsa a `0.00`.

F1-Macro es prácticamente igual (`0.45`) pero el reparto es distinto: LGBM gana en accuracy (`0.68` vs `0.63`) y en `Bad`; XGB gana en precision sobre `Neutral`. Las diferencias técnicas (level-wise vs leaf-wise) afectan más velocidad y distribución del esfuerzo entre clases que el F1-Macro final.


## 2.3 Stacking [0.7 Puntos]

### 2.3.1 Descripción del algoritmo [0.2 Puntos]

Describe con tus propias palabras cómo funciona el **Stacking**. Tu descripción debe cubrir los siguientes tres aspectos:

1. **Predicciones como features**: ¿Qué rol cumplen los modelos base? ¿Sobre qué datos generan sus predicciones para ser usadas por el meta-modelo? ¿Por qué se usa validación cruzada interna en lugar de predecir directamente sobre los datos de entrenamiento?
2. **Meta-modelo**: ¿Qué recibe como input el meta-modelo y qué aprende? ¿En qué se diferencia su rol del de los modelos base?
3. **Ventaja sobre selección simple**: ¿Por qué el Stacking puede superar a cualquier modelo base individual? ¿Qué aprovecha de la diversidad entre modelos?

> **Respuesta:**

**Respuesta:**

1. **Predicciones como features.** Los modelos base se entrenan sobre el dataset original y sus **predicciones** alimentan al meta-modelo. Para que sean honestas (sin contaminación por overfitting) se obtienen vía **validación cruzada interna**: cada base predice los folds que no vio. Sin CV, las predicciones sobre el propio train serían casi perfectas y el meta-modelo aprendería a confiar ciegamente en ellas.

2. **Meta-modelo.** Recibe la matriz de predicciones out-of-fold y aprende **cómo combinarlas**. A diferencia de los bases, no modela `X → y` sino `predicciones_base → y`.

3. **Ventaja.** El stacking **explota la diversidad**: si un modelo acierta donde otro falla, el meta-modelo aprende esa complementariedad. Por eso conviene mezclar familias con sesgos inductivos distintos (lineal, probabilístico, árboles).


---

### 2.3.2 Implementación [0.4 Puntos]

**Restricciones:**
- Mínimo **3 modelos base distintos**.
- Solo clasificadores básicos de scikit-learn: `LogisticRegression`, `MultinomialNB`, `SGDClassifier`, `DecisionTreeClassifier`, etc. **No se permiten modelos de ensamblaje** (`RandomForest`, `XGBoost`, `LightGBM`).
- El meta-modelo es de libre elección. Justifica tu elección.

**To-do:**
- [ ] Definir al menos 3 modelos base (solo clasificadores básicos de scikit-learn).
- [ ] Elegir un meta-modelo.
- [ ] Pipeline con `StackingClassifier(cv=3, n_jobs=-1)`.
- [ ] Reportar `classification_report`.

In [ ]:
estimators = [
    ("lr", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
    ("nb", MultinomialNB()),
    ("dt", DecisionTreeClassifier(random_state=RANDOM_STATE)),
]

pipe_stack = Pipeline(
    [
        ("features", preprocessing_transformer),
        (
            "clf",
            StackingClassifier(
                estimators=estimators,
                final_estimator=LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
                cv=3,
                n_jobs=-1,
            ),
        ),
    ]
)

pipe_stack.fit(X_train, y_train)
y_pred_stack = pipe_stack.predict(X_test)

print("=== StackingClassifier ===")
print(classification_report(y_test, y_pred_stack))

=== StackingClassifier ===
              precision    recall  f1-score   support

         Bad       0.56      0.52      0.54        86
        Good       0.68      0.82      0.74       148
     Neutral       0.00      0.00      0.00        23

    accuracy                           0.65       257
   macro avg       0.42      0.45      0.43       257
weighted avg       0.58      0.65      0.61       257



### 2.3.3 Pregunta de Cierre [0.1 Puntos]

**Pregunta:** Justifica la elección de tus modelos base y meta-modelo: ¿por qué los elegiste y qué aporta cada uno? ¿El Stacking mejoró respecto a los modelos base individuales de las secciones anteriores? ¿A qué atribuyes ese resultado considerando cómo funciona el algoritmo?

> **Respuesta:**

**Respuesta:**

**Modelos base** (diversidad de sesgos):
- `LogisticRegression`: frontera lineal, robusta con muchas columnas de BoW.
- `MultinomialNB`: probabilístico, pensado para conteos discretos del `CountVectorizer`.
- `DecisionTreeClassifier`: capta interacciones no lineales sobre los scores numéricos.

**Meta-modelo:** otra `LogisticRegression`. Recibe pocas features (una por base × clase) en escalas comparables, así que un modelo lineal basta y evita sobreajustar.

**Resultados.** Stacking obtiene accuracy `0.65` y F1-Macro `0.43`. Mejora claramente al árbol del baseline (`0.51` / `0.39`) y al RandomForest (`0.63` / `0.38`), pero queda **por debajo de Boosting** en F1-Macro (`0.45`). `Neutral` sigue colapsando a `0.00`, igual que en RF y LGBM: el desbalance no se resuelve por la vía del ensamblaje.


---

# 3. Optimización de Hiperparámetros con Optuna [1.5 Puntos]

<p align="center">
  <img src="https://media.giphy.com/media/3oKIPEqDGUULpEU0aQ/giphy.gif" width="300">
</p>

Hasta ahora hemos usado hiperparámetros por defecto. En esta sección optimizaremos **LightGBM** con **Optuna**, una librería de optimización bayesiana más eficiente que `GridSearchCV` porque:

- **Samplers inteligentes (TPE)**: usa el historial de trials para proponer mejores valores.
- **Pruning**: abandona trials poco prometedores antes de completarlos.
- **Visualizaciones interactivas**: analiza el espacio de hiperparámetros con `optuna-dashboard`.

Los resultados del estudio se persistirán en una base de datos SQLite, lo que permite explorarlos en tiempo real con el dashboard.

### 3.1 Descripción: Hiperparámetros y Optimización [0.2 Puntos]

Antes de implementar la optimización, reflexiona sobre los siguientes conceptos:

1. **¿Qué son los hiperparámetros?** ¿En qué se diferencian de los parámetros que el modelo aprende durante el entrenamiento?
2. **¿Por qué es importante optimizarlos?** ¿Qué consecuencias puede tener usar hiperparámetros por defecto?
3. **¿Qué es `GridSearchCV`?** Describe cómo funciona su mecanismo de búsqueda y cuál es su principal limitación cuando el espacio de hiperparámetros es grande.
4. **¿Por qué se evalúa cada trial con validación cruzada en lugar de usar directamente el conjunto de test?** ¿Qué problema introduce usar el test para seleccionar hiperparámetros? ¿Cómo podríamos reemplazar la CV por un conjunto de validación separado, y cuáles serían las ventajas y desventajas de ese enfoque?
5. **¿Cómo mejoran los samplers inteligentes la limitación de `GridSearchCV`?** ¿Por qué un sampler como TPE puede ser más eficiente que una búsqueda exhaustiva?

> **Respuesta:**

**Respuesta:**

1. **Hiperparámetros vs parámetros.** Los hiperparámetros se fijan **antes** del entrenamiento y controlan el comportamiento del modelo (estructura, regularización, learning rate). Los parámetros (pesos, splits) los aprende el modelo desde los datos.

2. **¿Por qué optimizar?** Usar defaults suele dejar el modelo subóptimo: o se subentrena o se sobreajusta. Los hiperparámetros impactan directo en generalización.

3. **`GridSearchCV`.** Búsqueda **exhaustiva** sobre una grilla predefinida, evaluando cada combinación con CV. Su límite es el costo: el número de combinaciones crece exponencialmente con la cantidad de hiperparámetros y rangos.

4. **CV vs test.** Cada trial se evalúa con CV para no contaminar el test, que debe quedar exclusivamente para la evaluación final. Si se usara el test para elegir hiperparámetros, su estimación dejaría de ser honesta. Una alternativa es un conjunto de validación separado: más rápido (una sola partición) pero **menos robusto** (más sensible a la suerte de la división).

5. **Samplers inteligentes.** TPE usa el historial de trials para concentrar la búsqueda en zonas prometedoras del espacio. Llega a soluciones comparables a `GridSearchCV` con muchos menos trials.


### 3.2 Definición del Espacio de Búsqueda y Función Objetivo [0.5 Puntos]

La función `objective(trial)` recibe un trial de Optuna, define los hiperparámetros a probar y devuelve la métrica a optimizar (F1-Macro).

| Hiperparámetro | Tipo | Rango | Descripción |
|----------------|------|-------|-------------|
| `num_leaves` | `suggest_int` | [20, 200] | Número máximo de hojas por árbol; controla la complejidad del modelo. |
| `learning_rate` | `suggest_float` (log) | [1e-3, 0.3] | Tasa de aprendizaje; cuánto contribuye cada árbol a la predicción final. |
| `max_depth` | `suggest_int` | [3, 12] | Profundidad máxima de cada árbol; limita la capacidad de memorización. |
| `min_child_samples` | `suggest_int` | [5, 100] | Mínimo de muestras requeridas en una hoja; regulariza contra sobreajuste. |
| `subsample` | `suggest_float` | [0.5, 1.0] | Fracción de filas muestreadas aleatoriamente para entrenar cada árbol. |
| `colsample_bytree` | `suggest_float` | [0.5, 1.0] | Fracción de features muestreadas aleatoriamente para cada árbol. |
| `reg_alpha` | `suggest_float` (log) | [1e-8, 10.0] | Regularización L1 sobre los pesos de las hojas. |
| `reg_lambda` | `suggest_float` (log) | [1e-8, 10.0] | Regularización L2 sobre los pesos de las hojas. |

**To-do:**
- [ ] Implementar `objective(trial)` con el espacio de búsqueda indicado.
- [ ] Usar `StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)`.
- [ ] Retornar la media del F1-Macro sobre los 3 folds.

In [ ]:
def objective(trial):
    params = {
        "num_leaves": trial.suggest_int("num_leaves", 20, 200),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    }

    pipe = Pipeline(
        [
            ("features", preprocessing_transformer),
            ("clf", LGBMClassifier(**params, random_state=RANDOM_STATE, verbose=-1)),
        ]
    )

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="f1_macro")
    return scores.mean()

### 3.3 Ejecución del Estudio de Optimización [0.4 Puntos]

Crea el estudio Optuna con 50 trials usando `TPESampler`. El estudio se guarda en el storage SQLite configurado anteriormente para poder explorarlo con `optuna-dashboard`.

**To-do:**
- [ ] Crear el estudio con `direction="maximize"`, `TPESampler(seed=RANDOM_STATE)` y el `storage` configurado.
- [ ] Ejecutar `study.optimize` con `n_trials=50` y `show_progress_bar=True`
- [ ] Imprimir el mejor valor y los mejores hiperparámetros.

In [ ]:
storage = optuna.storages.RDBStorage("sqlite:///optuna_lab7.db")
study = optuna.create_study(
    direction="maximize",
    study_name="lgbm-comics",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    storage=storage,
    load_if_exists=True,
)

remaining = 50 - len(study.trials)
if remaining > 0:
    study.optimize(objective, n_trials=remaining, show_progress_bar=True)

print(f"Mejor F1-Macro: {study.best_value:.4f}")
print("\nMejores hiperparámetros:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

### 3.4 Visualizaciones de Optuna [0.2 Puntos]

Optuna provee visualizaciones interactivas para analizar el proceso de optimización.

**To-do:**
- [ ] Graficar el historial de optimización (`plot_optimization_history`).
- [ ] Graficar la importancia de hiperparámetros (`plot_param_importances`).
- [ ] Graficar una visualización del comportamiento de los hiperparámetros (`plot_parallel_coordinate`)

In [ ]:
fig_history = plot_optimization_history(study)
fig_history.show()

fig_importances = plot_param_importances(study)
fig_importances.show()

fig_parallel = plot_parallel_coordinate(study)
fig_parallel.show()

### 3.5 Comparar Modelo Base con Mejores Hiperparámetros [0.1 Puntos]

**To-do:**
- [ ] Reentrenar un pipeline `pipe_lgbm_opt` con `study.best_params`.
- [ ] Comparar F1-Macro entre `pipe_lgbm` (defaults) y `pipe_lgbm_opt` (optimizado).

In [ ]:
pipe_lgbm_opt = Pipeline(
    [
        ("features", preprocessing_transformer),
        (
            "clf",
            LGBMClassifier(
                **study.best_params,
                random_state=RANDOM_STATE,
                verbose=-1,
                n_jobs=-1,
            ),
        ),
    ]
)

pipe_lgbm_opt.fit(X_train, y_train)
y_pred_lgbm_opt = pipe_lgbm_opt.predict(X_test)

print("=== LightGBM Default ===")
print(classification_report(y_test, y_pred_lgbm))

print("=== LightGBM Optimizado ===")
print(classification_report(y_test, y_pred_lgbm_opt))

f1_lgbm_default = f1_score(y_test, y_pred_lgbm, average="macro")
f1_lgbm_opt = f1_score(y_test, y_pred_lgbm_opt, average="macro")

print(f"F1-Macro LightGBM default: {f1_lgbm_default:.4f}")
print(f"F1-Macro LightGBM optimizado: {f1_lgbm_opt:.4f}")

=== LightGBM Default ===
              precision    recall  f1-score   support

         Bad       0.60      0.56      0.58        86
        Good       0.72      0.86      0.78       148
     Neutral       0.00      0.00      0.00        23

    accuracy                           0.68       257
   macro avg       0.44      0.47      0.45       257
weighted avg       0.61      0.68      0.64       257

=== LightGBM Optimizado ===
              precision    recall  f1-score   support

         Bad       0.59      0.55      0.57        86
        Good       0.71      0.84      0.77       148
     Neutral       1.00      0.09      0.16        23

    accuracy                           0.68       257
   macro avg       0.77      0.49      0.50       257
weighted avg       0.70      0.68      0.65       257

F1-Macro LightGBM default: 0.4533
F1-Macro LightGBM optimizado: 0.5001


### 3.6 Preguntas de reflexión [0.1 Puntos]

1. Según `plot_param_importances`, ¿qué hiperparámetro tuvo más impacto? ¿Tiene sentido dado tu conocimiento sobre algoritmos de Boosting?
2. ¿Qué información adicional te entregaron los gráficos de optuna comparado con solo mirar los logs de texto del estudio? ¿Qué pasa si en el gráfico interactivo `plot_parallel_coordinate` seleccionas con el mouse el rango específico con mejores resultados, hay patrones interesantes? - Hint: hace drag con el puntero sobre la dimensión de _Objective Value_ para seleccionar las mejores observaciones.

**Respuesta:**

1. El hiperparámetro con mayor impacto fue **`learning_rate`** (importancia ≈ 0.53), seguido de `reg_alpha` (≈ 0.37). Tiene sentido: `learning_rate` controla cuánto contribuye cada árbol al ensamble, así que regula directamente el balance entre subajuste y sobreajuste; `reg_alpha` (L1) controla complejidad.

2. Los gráficos muestran patrones que los logs no: la **evolución temporal** del estudio (history alcanza ~0.461 escalonadamente), la **importancia relativa** entre hiperparámetros, y las **interacciones**. En `plot_parallel_coordinate`, seleccionando los mejores trials por objetivo se observa que tienden a tener `learning_rate` relativamente alto, profundidad moderada y regularización positiva.


# 4. Interpretabilidad [1 Punto]

<p align="center">
  <img src="https://media.giphy.com/media/xT9IgzoKnwFNmISR8I/giphy.gif" width="300">
</p>

En esta sección interpretaremos las predicciones de LightGBM. Para obtener interpretaciones claras y directas, entrenaremos un **modelo auxiliar de LGBM usando únicamente los 6 atributos numéricos** (scores de habilidades), sin incluir features de texto.

Esto nos permite:
- Aplicar PFI y SHAP sin complicaciones de alta dimensionalidad.
- Obtener explicaciones semánticamente significativas (fuerza, inteligencia, velocidad...).

> **Nota:** En esta sección entrenaremos un modelo auxiliar para interpretabilidad con menos atributos. Usaremos un modelo entrenado desde 0, no el mejor seleccionado anteriormente.

In [ ]:
NUMERICAL_FEATURES = [
    "intelligence_score",
    "strength_score",
    "speed_score",
    "durability_score",
    "power_score",
    "combat_score",
]

X_train_num = X_train[NUMERICAL_FEATURES]
X_test_num = X_test[NUMERICAL_FEATURES]

lgbm_interp = LGBMClassifier(
    random_state=RANDOM_STATE,
    verbose=-1,
    n_jobs=-1,
)

lgbm_interp.fit(X_train_num, y_train)

## 4.1 Permutation Feature Importance (PFI) [0.4 Puntos]

### 4.1.1 Descripción [0.1 Puntos]

Responde las siguientes preguntas con tus propias palabras:

1. **¿Cómo funciona la Permutation Feature Importance?** ¿Qué se permuta exactamente y cómo se mide el impacto en el rendimiento del modelo?
2. **¿Por qué PFI es preferible a la importancia nativa de los árboles de decisión?** ¿Qué sesgo tiene la importancia nativa que PFI evita?

> **Respuesta:**

**Respuesta:**

1. **PFI.** Se permutan los valores de una feature (rompiendo su relación con `y`) y se mide cuánto cae la métrica del modelo ya entrenado. Si la caída es grande, la feature aportaba señal.

2. **PFI vs importancia nativa de árboles.** La nativa cuenta cuántos splits usan cada feature ponderados por la reducción de impureza, lo que **sesga** la importancia hacia features con muchos valores únicos (más oportunidades de partir). PFI mide impacto sobre el desempeño real del modelo y evita ese sesgo.


---

### 4.1.2 Implementación [0.2 Puntos]

**To-do:**
- [ ] Calcular `permutation_importance` con `n_repeats=30` y `scoring="f1_macro"`.
- [ ] Graficar un boxplot horizontal con plotly (`px.bar` con `orientation='h'`) con la importancia y varianza de cada feature.

In [28]:
pfi = permutation_importance(
    lgbm_interp,
    X_test_num,
    y_test,
    n_repeats=30,
    random_state=RANDOM_STATE,
    scoring="f1_macro",
)

pfi_df = pd.DataFrame(
    {
        "feature": NUMERICAL_FEATURES,
        "importance_mean": pfi.importances_mean,
        "importance_std": pfi.importances_std,
    }
).sort_values("importance_mean", ascending=True)

fig = px.bar(
    pfi_df,
    x="importance_mean",
    y="feature",
    error_x="importance_std",
    orientation="h",
    title="Permutation Feature Importance",
)

fig.show()

pfi_df.sort_values("importance_mean", ascending=False)

,feature,importance_mean,importance_std
0,intelligence_score,-0.006014,0.020063
1,strength_score,-0.012777,0.023514
5,combat_score,-0.018331,0.015188
3,durability_score,-0.021454,0.020887
4,power_score,-0.025023,0.018797
2,speed_score,-0.033306,0.019938


### 4.1.3 Pregunta de Cierre [0.1 Puntos]

**Pregunta:** Según el gráfico de PFI, ¿qué feature tuvo mayor importancia? ¿Tiene sentido intuitivo dado el contexto del problema (clasificar la alineación de personajes de cómics)? ¿Hay alguna feature cuya importancia te sorprenda?

> **Respuesta:**

**Respuesta:**

Ninguna feature tiene importancia positiva clara. La menos mala es `intelligence_score` (`-0.0060`), seguida de `strength`, `combat`, `durability`, `power` y `speed` (`-0.0333`). Permutar las features no empeora el modelo — incluso a veces lo mejora levemente — lo que significa que **los scores numéricos por sí solos no aportan señal predictiva fuerte**.

Tiene sentido para el problema: la alineación de un personaje depende mucho más de su historia narrativa que de cuánta fuerza o velocidad tenga. El modelo auxiliar sirve para interpretar, pero el grueso de la señal está en `history_text`.


## 4.2 SHAP (SHapley Additive exPlanations) [0.6 Puntos]

### 4.2.1 Descripción [0.2 Puntos]

Responde las siguientes preguntas con tus propias palabras:

1. **¿Qué miden los SHAP values?** ¿En qué se diferencian de una medida de importancia global como PFI?
2. **¿Qué representa el `base_value` en un `waterfall_plot` de SHAP?** ¿Por qué es el punto de partida para interpretar una predicción individual?
3. **¿Por qué usar `TreeExplainer` para modelos basados en árboles?** ¿Qué ventaja ofrece respecto a un explainer genérico?

> **Respuesta:**

**Respuesta:**

1. **SHAP values.** Miden el aporte de cada feature a la predicción de **una instancia específica**. PFI entrega una importancia global (un número por feature); SHAP da un valor por feature **y por observación**, lo que permite explicar predicciones individuales.

2. **`base_value` del `waterfall_plot`.** Es la predicción promedio del modelo sobre el dataset (el valor "neutro" antes de ver los features de la instancia). Desde ahí, cada feature suma o resta para llegar a la predicción final del caso particular.

3. **`TreeExplainer`.** Está diseñado para modelos basados en árboles (LightGBM, XGBoost, RF). Aprovecha la estructura interna para calcular SHAP values de forma **exacta y eficiente**, en vez de aproximarlos con muchas evaluaciones del modelo como hace un explainer genérico.


---

### 4.2.2 Implementación [0.2 Puntos]

**To-do:**
- [ ] Crear un `shap.TreeExplainer(lgbm_interp)` y calcular `shap_values` sobre `X_test_num`.
- [ ] `summary_plot` para ver importancia global y dirección del efecto (para la clase "Good").
- [ ] `waterfall_plot` para la predicción de una instancia cualquiera de `X_test_num`.

In [ ]:
explainer = shap.TreeExplainer(lgbm_interp)
shap_values = explainer.shap_values(X_test_num)

class_names = list(lgbm_interp.classes_)
good_idx = class_names.index("Good")

shap_values_good = shap_values[:, :, good_idx]
expected_value_good = explainer.expected_value[good_idx]

shap.summary_plot(
    shap_values_good,
    X_test_num,
    feature_names=NUMERICAL_FEATURES,
)

idx = 0

shap.waterfall_plot(
    shap.Explanation(
        values=shap_values_good[idx],
        base_values=expected_value_good,
        data=X_test_num.iloc[idx],
        feature_names=NUMERICAL_FEATURES,
    )
)

### 4.2.3 Pregunta de Cierre [0.2 Puntos]

1. ¿Qué diferencia existe entre Permutation Feature Importance y los SHAP values como medida de importancia de features?
2. Según el `waterfall_plot`, ¿qué features fueron las que más empujaron la predicción hacia su clase? Investiga el personaje seleccionado: ¿Tiene sentido dado su historia en los cómics?

> **Respuesta:**

**Respuesta:**

1. **PFI vs SHAP.** PFI es **global** y mide impacto sobre el rendimiento al permutar features. SHAP es **local**: explica cada predicción individual indicando cuánto y en qué dirección empujó cada feature. SHAP se puede agregar para una vista global; PFI no se puede desagregar.

2. **`waterfall_plot`.** Para la instancia mostrada (clase `Good`), las features con mayor magnitud son `durability_score` (negativa), `speed_score` e `intelligence_score`. El modelo se apoya principalmente en habilidades físicas para mover la predicción desde el `base_value`. Tiene sentido parcialmente: las habilidades dicen algo de la alineación, pero el modelo auxiliar no usa la historia textual — que probablemente contiene más señal sobre si es bueno, malo o neutral.


---

# 5. Predicción de Personajes No Etiquetados [0.5 Puntos]

<p align="center">
  <img src="https://pbs.twimg.com/media/DolotxUUYAAbg7f.jpg" width="350">
</p>

¡Llegó el momento de predecir `Vergil`, `Gorilla Girl` y `Bat-Cow`!

Usaremos el **mejor modelo** obtenido en la sección 3 (`pipe_lgbm_opt`) para predecir la alineación de los personajes no etiquetados.

**Nota:** Recuerda eliminar los NaN en `history_text` antes de predecir.

### 5.0 Predicción [0.2 Puntos]

**To-do:**
- [ ] Usar `pipe_lgbm_opt` para predecir `alignment` en `df_comics_no_label` (recuerda eliminar NaN en `history_text`).
- [ ] Filtrar y mostrar resultados para `Vergil`, `Gorilla Girl` y `Bat-Cow`.

In [39]:
personajes_objetivo = ["Vergil", "Gorilla Girl", "Batcow"]

df_pred = df_comics_no_label.dropna(subset=["history_text"]).copy()

df_pred["alignment_pred"] = pipe_lgbm_opt.predict(df_pred[FEATURES])

df_personajes = df_pred.loc[
    df_pred["name"].isin(personajes_objetivo),
    ["name", "alignment_pred"],
].drop_duplicates()

df_personajes

,name,alignment_pred
122,Batcow,Good
529,Gorilla Girl,Good
1368,Vergil,Good


Para conectar con la sección 4, generamos el `waterfall_plot` de Batcow usando `lgbm_interp` (modelo auxiliar entrenado solo con los 6 scores numéricos).


In [ ]:
row_batcow = df_pred.loc[df_pred["name"] == "Batcow", NUMERICAL_FEATURES].iloc[[0]]
shap_values_batcow = explainer.shap_values(row_batcow)

shap.waterfall_plot(
    shap.Explanation(
        values=shap_values_batcow[0, :, good_idx],
        base_values=expected_value_good,
        data=row_batcow.iloc[0],
        feature_names=NUMERICAL_FEATURES,
    )
)

### 5.1 Análisis de Predicciones [0.3 Puntos]

**Pregunta:** Comenta las predicciones obtenidas para `Vergil`, `Gorilla Girl` y `Bat-Cow`:

1. ¿Las predicciones te parecen razonables según lo que conoces (o puedes inferir) de estos personajes?
2. Conecta con la sección 4: ¿qué features numéricas habrían influido más en la predicción de **Bat-Cow** según el `waterfall_plot`? ¿Es consistente con la predicción obtenida aquí?

> **Respuesta:**

**Respuesta:**

1. El modelo predice `Good` para los tres. **Batcow** encaja: el debate del enunciado se resuelve a favor del profesor (no nace mala). **Gorilla Girl** también es razonable: por su historia y rol como heroína de Marvel. **Vergil** es el caso más cuestionable: en Devil May Cry suele ser antagonista, pero el modelo aprende patrones de texto, no moralidad explícita, así que la predicción depende de cómo aparece descrito en su `history_text`.

2. El `waterfall_plot` específico de Batcow (sección 5 arriba) muestra qué features numéricas empujan su predicción y desde qué `base_value` parten. Sus scores son bajos en general, así que su clasificación final como `Good` se sostiene principalmente sobre el `history_text` del pipeline completo `pipe_lgbm_opt`, más que sobre los scores numéricos del modelo auxiliar. Consistente con 4.1: los scores aislados aportan poca señal predictiva.


# Conclusión

¡Eso ha sido todo para el lab de hoy! Recuerden que el laboratorio tiene un plazo de entrega de una semana y que **los días de atraso no se pueden utilizar para entregas de lab, solo para tareas**. Cualquier duda del laboratorio, no duden en contactarnos por mail o U-cursos.

<p align="center">
  <img src="https://media1.tenor.com/images/fb5bf7cc5a4acb91b4177672886a88ba/tenor.gif?itemid=5591338">
</p>